# Evaluate a trained detector on whole photos

Scores a model trained by `train.ipynb` on the 19 validation photos (43 butts), which were labelled without any
detector suggestions on screen and never used in training. The evaluation harness runs the model on each full
photo the way it would run on a robot's camera frame: 640 px tiles, boxes mapped back and merged across tiles.

How a detection is scored:

- A butt is **found** when a detection at or above the threshold has its centre inside the butt's box, or
  overlaps it with IoU ≥ 0.3.
- A **false positive** is a detection on no labelled box. Several detections on one butt are not penalised.
- Boxes labelled `maybe` (objects the annotator was unsure of) count neither way.

The result is reported at the operating threshold (0.25, Ultralytics' default) and as a curve over every
threshold: recall against false positives per photo, interactive, with the published reference models for
comparison.

**On Kaggle:** add the dataset `lorinc/butt-field-2026-10-01` and the output of your training notebook as
inputs, and turn internet on. A GPU is optional: on a CPU the 19 photos take a few minutes.

## Settings

In [ ]:
RUN_CONFIG = None     # the training notebook's run_config.json; None finds it under /kaggle/input or ./work
WEIGHTS = None        # trained weights; None: train/weights/last.pt next to RUN_CONFIG
REPO = 'https://github.com/lorinc/litter-detection-pipeline.git'
COMMIT = 'main'       # branch, tag or commit of the code to clone
DATA = None           # dataset folder (images/, labels/, split.json, meta.csv); None finds it under /kaggle/input
WORK = None           # output folder; None: /kaggle/working on Kaggle, ./work elsewhere
REFERENCES = True     # draw the published reference models on the curve as well

## Code, dataset and dependencies

In [ ]:
import glob, json, os, pathlib, subprocess, sys

def sh(cmd, cwd=None, env=None):
    """Run a shell command, streaming its output into the notebook; stop on failure."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'exit {p.returncode}: {cmd}')

def show(text):
    """Markdown in a notebook, plain text elsewhere."""
    try:
        from IPython.display import Markdown, display
        display(Markdown(text))
    except ImportError:
        print(text)

ON_KAGGLE = os.path.isdir('/kaggle/input')
if os.path.isfile('evaluate.py') and os.path.isfile('curve_plot.py'):
    CODE = pathlib.Path.cwd()  # running inside a clone's pipeline/ folder
else:
    clone = pathlib.Path('/tmp/litter-detection-pipeline')
    if not clone.exists():
        sh(f'git clone -q {REPO} {clone}')
    sh(f'git -C {clone} checkout -q {COMMIT}')
    CODE = clone / 'pipeline'
sys.path.insert(0, str(CODE))
if DATA is None:
    found = glob.glob('/kaggle/input/**/split.json', recursive=True)
    assert found, 'dataset not found: add lorinc/butt-field-2026-10-01 as input, or set DATA'
    DATA = os.path.dirname(found[0])
WORK = pathlib.Path(WORK or ('/kaggle/working' if ON_KAGGLE else 'work')).resolve()
if RUN_CONFIG is None:
    found = glob.glob('/kaggle/input/**/run_config.json', recursive=True) + glob.glob(str(WORK / '*/run_config.json'))
    assert len(found) == 1, f'set RUN_CONFIG: found {len(found)} run_config.json files {found}'
    RUN_CONFIG = found[0]
RUN_CONFIG = pathlib.Path(RUN_CONFIG)
WEIGHTS = pathlib.Path(WEIGHTS or RUN_CONFIG.parent / 'train/weights/last.pt')
assert WEIGHTS.exists(), f'no weights at {WEIGHTS}; set WEIGHTS'

cfg = json.load(open(RUN_CONFIG))
version = cfg['train']['ultralytics']
try:
    import ultralytics
    installed = ultralytics.__version__
except ImportError:
    installed = None
if installed != version:
    sh(f'{sys.executable} -m pip install -q ultralytics=={version}')
try:
    import plotly
except ImportError:
    sh(f'{sys.executable} -m pip install -q plotly==5.24.1')
print(f'code    {CODE}')
print(f'data    {DATA}')
print(f'run     {RUN_CONFIG}')
print(f'weights {WEIGHTS}')

## Evaluation configuration

The run's configuration with the weights path set to where the weights are now, saved as `eval_config.json`.

In [ ]:
RUN = next(n for n in cfg['eval']['detectors'] if n != 'owlv2')
out = WORK / RUN
out.mkdir(parents=True, exist_ok=True)
cfg['eval']['detectors'][RUN]['weights'] = str(WEIGHTS.resolve())
CONFIG = out / 'eval_config.json'
json.dump(cfg, open(CONFIG, 'w'), indent=2)
print(f'{RUN}: {cfg["eval"]["detectors"][RUN]}')

## Score the validation photos

Detections are cached per photo under `cache/`, so a re-run only re-scores.

In [ ]:
sh(f'{sys.executable} evaluate.py {RUN} --config {CONFIG} --data {DATA} --cache {WORK / "cache"} --out {out}',
   cwd=CODE)
result = json.load(open(out / f'{RUN}_val.json'))

## At the operating threshold

Recall intervals are Wilson 95% intervals over butts; the false-positive interval comes from resampling whole
photos. With 43 butts, one butt is about 2 points of recall.

In [ ]:
op = result['operating_point']
lo, hi = op['recall_ci95']
flo, fhi = op['fp_per_frame_ci95']
show(f'''| {RUN}, threshold {result["threshold"]} | |
|---|---|
| Recall | {op["recall"]:.1%} ({op["found"]} of {op["butts"]} butts; {lo:.1%}–{hi:.1%}) |
| False positives per photo | {op["fp_per_frame"]:.2f} ({flo:.2f}–{fhi:.2f}) |
| Precision | {op["precision"]:.1%} |
| Boxes per photo | {op["boxes_per_frame"]:.2f} |
| Time per photo, this machine | {result["speed"]["ms_per_frame_mean"] / 1000:.1f} s |
| Parameters; compute per photo | {result["compute"]["params"] / 1e6:.1f} M; {result["compute"]["gflops_per_frame"]:,.0f} GFLOPs |''')

rows = '\n'.join(f'| {b["bin"]} px | {b["n"]} | {b["found"]} | ' + (f'{b["recall"]:.0%}' if b['recall'] is not None else '–') + ' |'
                 for b in result['by_size_bin'] if b['n'])
show('**By butt size** (long side of the box, native px)\n\n| Size | Butts | Found | Recall |\n|---|---|---|---|\n' + rows)
rows = '\n'.join(f'| {sh_} | {v["frames"]} | {v["found"]} of {v["butts"]} | {v["fp_per_frame"]:.2f} |'
                 for sh_, v in result['by_shot'].items())
show('**By shot type**\n\n| Shot | Photos | Butts found | False positives per photo |\n|---|---|---|---|\n' + rows)

## Recall against false positives per photo

Each point of a curve is one threshold; hover for its value. Lower thresholds find more butts and raise more
false alarms. The dots are the operating thresholds. The reference curves are the published results on the same
19 photos: the zero-shot OWLv2 baseline and the two cross-validation fold 0 models of the web demo. The curves
carry no intervals, and every photo comes from one capture session.

In [ ]:
import curve_plot
results = [(RUN, result)]
if REFERENCES:
    refs = CODE.parent / 'docs/results'
    for f, label in [('owlv2_val.json', 'OWLv2, zero-shot (reference)'),
                     ('04-cv-hard-negatives_f0_val.json', 'YOLO11s, 04 fold 0 (reference)'),
                     ('06-cv-yolo11n_f0_val.json', 'YOLO11n, 06 fold 0 (reference)')]:
        r = json.load(open(refs / f))
        if r['labels_sha256'] == result['labels_sha256']:
            results.append((label, r))
        else:
            print(f'{label} left out: scored on different labels than this dataset version')
fig = curve_plot.figure(results)
fig.write_html(out / 'recall_fp_curve.html', include_plotlyjs='cdn')
fig.show()
print(f'saved {out / "recall_fp_curve.html"} and {out / f"{RUN}_val.json"}')